Here is a **clean, deduplicated summary** of your full benchmarking design, organized into unique functional components without repetition:

---

## 1. Unified Benchmarking Harness

* Wrap both `generate_society_discrete()` and `generate_society_density()`
* Capture:

  * Execution time
  * Peak memory usage
  * Output tensors (traits, wealth, influence, etc.)
  * Metadata for downstream structural analysis
* Store results in a centralized DataFrame or structured log

---

## 2. Multi-Dimensional Parameter Grid

Grid search across:

* `num_agents`: 1,000 → 100,000
* `mutation_temperature`: 0.0 → 1.0 (step 0.1)
* `use_power_law_influence`: True / False

Purpose:

* Produce hundreds of controlled simulation environments
* Enable scalability and robustness testing

---

## 3. Core Quantitative Metrics

### • Gini Coefficient (Wealth Inequality)

* Measures structural resource concentration
* Compare:

  * Structural inequality baseline
  * Power-law influence effects
  * Discrete vs Density-based outputs

### • Shannon Entropy (Role & Region Distributions)

* Measures social diversity and distributional smoothness
* Detects “role blurring” in density-based model
* Higher entropy → more fluid social transitions

### • DNA Drift (Centroid Distance)

* Euclidean distance between:

  * Generated agents
  * Original archetype DNA
* Quantifies identity diffusion at different temperatures
* Measures how much social identity erodes

---

## 4. Boundary Clamping Verification

Extreme test at `mutation_temperature = 1.0`

Validate:

* Non-wealth traits ∈ [-1, 1]
* Wealth dimensions respect defined bounds
* Zero violations across tensors

Purpose:

* Ensure stability under maximum entropy conditions

---

## 5. Recursive Stress-Test Harness

* Iterates through full parameter grid
* Automates nested combinations
* Logs metrics per configuration
* Designed to scale to 100k agents safely

---

## 6. Scalability Benchmark

Isolate performance of:

* `torch.cdist` (density-based)
* `np.random.choice` loop (discrete)

Measure:

* Execution time growth
* Memory scaling
* O(n) or O(n²) bottlenecks

Goal:

* Identify large-scale performance ceilings

---

## 7. Deterministic Reproducibility Test

For identical seeds:

* Run same configuration 5×
* Verify bitwise tensor equality
* Confirm:

  * Gaussian mixture sampling determinism
  * Mutation consistency
  * Seed integrity

---

## 8. Comparative Distribution Visualization

Generate:

* Wealth histograms
* Influence histograms
* 2D KDE plots

Side-by-side:

* Discrete vs Density-based
* With clear legends

Purpose:

* Visually detect:

  * Outliers
  * Heavy tails
  * Social smoothing
  * Concentration shifts

---

## 9. Role Blurring Analysis

Using entropy + visualizations:

* Determine if density-based model creates:

  * Transitional identities
  * Fewer hard categorical boundaries
  * More realistic social gradients

---

## 10. Final Model Evaluation Framework

Evaluate each model across:

* Structural inequality realism
* Natural outlier formation
* Trait stability
* Entropy balance
* Scalability
* Determinism
* Boundary safety

Deliver:

* Recommendation for final simulation engine
* Tradeoff summary (performance vs realism)

---

# Condensed High-Level View

Your benchmarking suite evaluates both models across:

1. Performance
2. Structural inequality
3. Entropy and diversity
4. Identity drift
5. Stability constraints
6. Determinism
7. Scalability
8. Distribution realism

---

In [ ]:
# =====================
# Imports
# =====================
import math
import numpy as np
import pandas as pd
import torch
import torch.distributions as dist
from dataclasses import dataclass
from typing import Dict, List


# ============================================================
# ===================== SCHEMA ===============================
# ============================================================

DIMENSIONS: List[str] = [
    "Wealth","Physical_Safety","Stability","Reputation","Fairness",
    "In_Group","Innovation","Freedom","Sanctity","Care",
    "Short_Term","Long_Term",
]

REGIONS: List[str] = ["US", "EU", "APAC", "Global_South"]
REGION_WEIGHTS = [0.15, 0.15, 0.35, 0.35]

REGION_ROLE_PROBS = {
    "US": [0.02, 0.45, 0.10, 0.20, 0.18, 0.05],
    "EU": [0.01, 0.48, 0.08, 0.25, 0.15, 0.03],
    "APAC": [0.005, 0.60, 0.05, 0.10, 0.20, 0.045],
    "Global_South": [0.001, 0.65, 0.02, 0.05, 0.25, 0.029],
}

ROLES: List[str] = ["CEO", "Worker", "Investor", "Retiree", "Student", "Politician"]

ROLE_TIER_MAP: Dict[str, int] = {
    "CEO": 0,
    "Investor": 1,
    "Politician": 0,
    "Worker": 2,
    "Retiree": 2,
    "Student": 2,
}

ROLE_STARTING_STATS = {
    "CEO": (50.0, 18.0),
    "Politician": (70.0, 25.0),
    "Investor": (25.0, 12.0),
    "Retiree": (10.0, 2.0),
    "Worker": (5.0, 1.0),
    "Student": (1.0, 0.6),
}

ARCHETYPES = {
    "CEO": {
        "exp": torch.tensor([0.8,0.1,0.3,0.9,0.1,0.4,0.9,0.8,0.1,0.1,0.7,0.9]),
        "big5": torch.tensor([0.8,0.9,0.8,0.3,0.3]),
    },
    "Worker": {
        "exp": torch.tensor([0.4,0.8,0.9,0.3,0.7,0.5,0.2,0.4,0.5,0.6,0.9,0.2]),
        "big5": torch.tensor([0.4,0.6,0.5,0.6,0.7]),
    },
    "Investor": {
        "exp": torch.tensor([0.9,0.1,0.2,0.6,0.2,0.3,0.8,0.8,0.0,0.1,0.8,0.6]),
        "big5": torch.tensor([0.7,0.8,0.5,0.2,0.4]),
    },
    "Retiree": {
        "exp": torch.tensor([0.3,0.9,1.0,0.2,0.5,0.8,-0.3,0.3,0.8,0.7,0.4,0.1]),
        "big5": torch.tensor([0.2,0.5,0.4,0.7,0.8]),
    },
    "Student": {
        "exp": torch.tensor([0.1,0.3,0.2,0.5,0.8,0.3,0.9,0.7,0.1,0.8,0.6,0.8]),
        "big5": torch.tensor([0.9,0.3,0.7,0.6,0.5]),
    },
    "Politician": {
        "exp": torch.tensor([0.6,0.5,0.8,1.0,0.4,0.9,0.3,0.6,0.6,0.5,1.0,0.2]),
        "big5": torch.tensor([0.6,0.7,0.9,0.5,0.4]),
    },
}


@dataclass
class SimConfig:
    seed: int = 42
    num_agents: int = 10000
    mutation_temperature: float = 0.7
    use_power_law_influence: bool = False

    # Generation mode
    generation_mode: str = "structured"  # "structured" | "bias_free"

    # Evolution
    enable_evolution: bool = True
    evolution_generations: int = 10
    inheritance_fraction: float = 0.7
    base_return_rate: float = 0.03
    influence_reinvestment_factor: float = 0.1
    shock_frequency: float = 0.1
    shock_magnitude: float = 0.2
    mobility_rate: float = 0.05

    # Structural entropy
    wealth_decay_rate: float = 0.01
    influence_decay_rate: float = 0.02
    redistribution_rate: float = 0.01

    # Dynamic roles
    use_dynamic_roles: bool = True
    role_temperature: float = 0.5
    elite_wealth_threshold: float = 0.95


# ============================================================
# ===================== GENERATOR ============================
# ============================================================

def apply_random_mutations(exposures, personalities, temperature, seed):
    if temperature <= 0.0:
        return exposures, personalities

    rng = torch.Generator()
    rng.manual_seed(seed + 999)

    n_agents = exposures.shape[0]
    num_dims = len(DIMENSIONS)

    mutant_mask = torch.rand(n_agents, generator=rng) < temperature
    mutant_indices = torch.where(mutant_mask)[0]
    num_mutants = mutant_indices.shape[0]

    if num_mutants == 0:
        return exposures, personalities

    num_changes = math.ceil(3 * temperature)

    for _ in range(num_changes):
        col_indices = torch.randint(0, num_dims, (num_mutants,), generator=rng)
        random_values = (torch.rand(num_mutants, generator=rng) * 2) - 1.0
        exposures[mutant_indices, col_indices] = random_values

    for _ in range(num_changes):
        col_indices = torch.randint(0, 5, (num_mutants,), generator=rng)
        personalities[mutant_indices, col_indices] = torch.rand(num_mutants, generator=rng)

    return exposures, personalities

In [ ]:
def generate_society(config: SimConfig):

    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    num_dims = len(DIMENSIONS)
    wealth_idx = DIMENSIONS.index("Wealth")

    regions_arr = np.random.choice(REGIONS, size=config.num_agents, p=REGION_WEIGHTS)
    roles_arr = np.empty(config.num_agents, dtype=object)

    for region in REGIONS:
        mask = regions_arr == region
        count = np.sum(mask)
        if count == 0:
            continue
        roles_arr[mask] = np.random.choice(
            ROLES, size=count, p=REGION_ROLE_PROBS[region]
        )

    tiers_arr = np.array([ROLE_TIER_MAP[r] for r in roles_arr])

    exposures = torch.zeros(config.num_agents, num_dims)
    personalities = torch.zeros(config.num_agents, 5)

    for role in ROLES:
        mask = roles_arr == role
        count = np.sum(mask)
        if count == 0:
            continue

        base_exp = ARCHETYPES[role]["exp"]
        base_big5 = ARCHETYPES[role]["big5"]

        exp_noise = torch.randn(count, num_dims) * 0.3
        exposures_with_noise = base_exp + exp_noise

        non_wealth_mask = torch.ones(num_dims, dtype=torch.bool)
        non_wealth_mask[wealth_idx] = False

        exposures_with_noise[:, non_wealth_mask] = torch.clamp(
            exposures_with_noise[:, non_wealth_mask], -1.0, 1.0
        )

        exposures[mask] = exposures_with_noise

        beta_scale = 10.0
        alpha = base_big5 * beta_scale + 1.0
        beta = (1.0 - base_big5) * beta_scale + 1.0

        alpha_exp = alpha.unsqueeze(0).expand(count, -1)
        beta_exp = beta.unsqueeze(0).expand(count, -1)

        personalities[mask] = dist.Beta(alpha_exp, beta_exp).sample()

    exposures, personalities = apply_random_mutations(
        exposures, personalities, config.mutation_temperature, config.seed
    )

    role_wealth = np.ones(config.num_agents)
    role_influence = np.ones(config.num_agents)

    for role, (w, i) in ROLE_STARTING_STATS.items():
        mask = roles_arr == role
        role_wealth[mask] = w
        role_influence[mask] = i

    if config.use_power_law_influence:
        pareto_w = np.clip(np.random.pareto(1.5, config.num_agents) + 1, 1, 50)
        pareto_i = np.clip(np.random.pareto(1.3, config.num_agents) + 1, 1, 20)
        role_wealth *= pareto_w
        role_influence *= pareto_i

    exposures[:, wealth_idx] *= torch.tensor(role_wealth).float()
    exposures[:, wealth_idx] = torch.clamp(exposures[:, wealth_idx], -100, 100)

    personalities = torch.clamp(personalities, 0.0, 1.0)

    df = pd.DataFrame({
        "Agent_ID": range(config.num_agents),
        "Role": roles_arr,
        "Region": regions_arr,
        "Tier": tiers_arr,
        "Influence": np.round(role_influence, 3),
    })

    print("Society Generated (in-memory)")
    return df, exposures, personalities


# ============================================================
# ===================== TEST RUN =============================
# ============================================================

conf = SimConfig(num_agents=10000, seed=69)
df, exposures, personalities = generate_society(conf)

df.head()

In [ ]:
# ============================================================
# DENSITY-BASED SOCIETY GENERATOR (GMM VERSION - ISOLATED)
# ============================================================

def generate_society_density_v2(
    density_config: SimConfig,
) -> tuple[pd.DataFrame, torch.Tensor, torch.Tensor]:
    """
    Density-based generator using Gaussian Mixture sampling.
    Fully isolated from other generator functions.
    """

    # Reproducibility
    torch.manual_seed(density_config.seed + 111)
    np.random.seed(density_config.seed + 111)

    print(f"Generating {density_config.num_agents} Agents via Density Field (GMM)...")

    num_dimensions_local = len(DIMENSIONS)
    wealth_dimension_index = DIMENSIONS.index("Wealth")

    # --------------------------------------------------------
    # 1. BUILD CONTINUOUS DENSITY LANDSCAPE
    # --------------------------------------------------------

    archetype_means_list = []
    for role_name in ROLES:
        combined_dna = torch.cat(
            [ARCHETYPES[role_name]["exp"], ARCHETYPES[role_name]["big5"]]
        )
        archetype_means_list.append(combined_dna)

    means_matrix = torch.stack(archetype_means_list)  # Shape: (roles, 17)

    # --------------------------------------------------------
    # 2. REGION-CONDITIONAL SAMPLING
    # --------------------------------------------------------

    region_labels_array = np.random.choice(
        REGIONS,
        size=density_config.num_agents,
        p=REGION_WEIGHTS,
    )

    traits_tensor = torch.zeros(density_config.num_agents, 17)

    for region_name in REGIONS:

        region_mask_np = region_labels_array == region_name
        region_mask_torch = torch.from_numpy(region_mask_np)
        region_count = region_mask_torch.sum().item()

        if region_count == 0:
            continue

        role_mix_weights = torch.tensor(REGION_ROLE_PROBS[region_name])

        component_ids = torch.multinomial(
            role_mix_weights,
            region_count,
            replacement=True,
        )

        noise_std_local = 0.15 + (density_config.mutation_temperature * 0.1)

        selected_means = means_matrix[component_ids]

        gaussian_noise = torch.randn(region_count, 17) * noise_std_local

        traits_tensor[region_mask_torch] = selected_means + gaussian_noise

    # --------------------------------------------------------
    # 3. SPLIT INTO EXPOSURES + PERSONALITY
    # --------------------------------------------------------

    exposures_density = traits_tensor[:, :num_dimensions_local]
    personalities_density = traits_tensor[:, num_dimensions_local:]

    exposures_density, personalities_density = apply_random_mutations(
        exposures_density,
        personalities_density,
        density_config.mutation_temperature,
        density_config.seed + 222,
    )

    # --------------------------------------------------------
    # 4. CLASSIFY BY CLOSEST ARCHETYPE
    # --------------------------------------------------------

    distance_matrix = torch.cdist(traits_tensor, means_matrix)
    closest_indices = torch.argmin(distance_matrix, dim=1)

    assigned_roles_array = np.array([ROLES[idx] for idx in closest_indices])
    tier_array_density = np.array([ROLE_TIER_MAP[r] for r in assigned_roles_array])

    # --------------------------------------------------------
    # 5. POWER LAW + WEALTH SCALING
    # --------------------------------------------------------

    wealth_base_array = np.array(
        [ROLE_STARTING_STATS[r][0] for r in assigned_roles_array]
    )

    influence_base_array = np.array(
        [ROLE_STARTING_STATS[r][1] for r in assigned_roles_array]
    )

    if density_config.use_power_law_influence:

        pareto_w_local = np.clip(
            np.random.pareto(1.5, density_config.num_agents) + 1.0,
            1.0,
            50.0,
        )

        pareto_i_local = np.clip(
            np.random.pareto(1.3, density_config.num_agents) + 1.0,
            1.0,
            20.0,
        )

        wealth_multiplier_tensor = torch.tensor(
            wealth_base_array * pareto_w_local
        ).float()

        influence_scores_density = influence_base_array * pareto_i_local

    else:

        wealth_multiplier_tensor = torch.tensor(wealth_base_array).float()
        influence_scores_density = influence_base_array

    exposures_density[:, wealth_dimension_index] *= wealth_multiplier_tensor

    exposures_density = torch.clamp(exposures_density, -1.0, 100.0)

    non_wealth_mask_local = torch.ones(num_dimensions_local, dtype=torch.bool)
    non_wealth_mask_local[wealth_dimension_index] = False

    exposures_density[:, non_wealth_mask_local] = torch.clamp(
        exposures_density[:, non_wealth_mask_local],
        -1.0,
        1.0,
    )

    personalities_density = torch.clamp(personalities_density, 0.0, 1.0)

    # --------------------------------------------------------
    # 6. METADATA
    # --------------------------------------------------------

    metadata_density_df = pd.DataFrame({
        "Agent_ID": range(density_config.num_agents),
        "Role": assigned_roles_array,
        "Region": region_labels_array,
        "Tier": tier_array_density,
        "Influence": np.round(influence_scores_density, 3),
    })

    print("Density-Based Society Generated (in-memory, isolated)")
    return metadata_density_df, exposures_density, personalities_density

In [ ]:
density_conf = SimConfig(
    num_agents=10000,
    seed=123,
    mutation_temperature=0.5,
    use_power_law_influence=True,
)

df_density, exposures_density, personalities_density = generate_society_density_v2(density_conf)

df_density.head()

In [ ]:
# ============================================================
# BENCHMARKING FRAMEWORK
# ============================================================

import time
import itertools
import tracemalloc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import entropy
from numpy.linalg import norm


# ============================================================
# 1. CORE METRICS
# ============================================================

def gini_coefficient(x: np.ndarray):
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * x)) / (n * np.sum(x) + 1e-9)


def shannon_entropy_distribution(arr):
    values, counts = np.unique(arr, return_counts=True)
    probs = counts / counts.sum()
    return entropy(probs)


def dna_drift_metric(traits_tensor):
    """
    Distance between agents and nearest archetype mean.
    """
    archetype_means = torch.stack([
        torch.cat([ARCHETYPES[r]["exp"], ARCHETYPES[r]["big5"]])
        for r in ROLES
    ])
    distances = torch.cdist(traits_tensor, archetype_means)
    return torch.mean(torch.min(distances, dim=1).values).item()


def boundary_violation_check(exposures_tensor, wealth_idx):
    non_wealth_mask = torch.ones(exposures_tensor.shape[1], dtype=torch.bool)
    non_wealth_mask[wealth_idx] = False

    non_wealth = exposures_tensor[:, non_wealth_mask]
    wealth = exposures_tensor[:, wealth_idx]

    violations_non_wealth = torch.sum((non_wealth < -1) | (non_wealth > 1)).item()
    violations_wealth = torch.sum((wealth < -100) | (wealth > 100)).item()

    return violations_non_wealth, violations_wealth


# ============================================================
# 2. PERFORMANCE HARNESS
# ============================================================

def run_generator_with_metrics(generator_fn, config):

    tracemalloc.start()
    start_time = time.time()

    df, exposures, personalities = generator_fn(config)

    execution_time = time.time() - start_time
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    traits = torch.cat([exposures, personalities], dim=1)

    wealth = exposures[:, DIMENSIONS.index("Wealth")].numpy()

    metrics = {
        "execution_time_sec": execution_time,
        "peak_memory_mb": peak / 1024**2,
        "gini_wealth": gini_coefficient(wealth),
        "role_entropy": shannon_entropy_distribution(df["Role"]),
        "region_entropy": shannon_entropy_distribution(df["Region"]),
        "dna_drift": dna_drift_metric(traits),
    }

    v_nonwealth, v_wealth = boundary_violation_check(
        exposures,
        DIMENSIONS.index("Wealth")
    )

    metrics["boundary_violations_nonwealth"] = v_nonwealth
    metrics["boundary_violations_wealth"] = v_wealth

    return metrics, df, exposures, personalities


# ============================================================
# 3. PARAMETER GRID TEST (Cases 1,2,5)
# ============================================================

def recursive_grid_benchmark(discrete_fn, density_fn):

    results = []

    agent_sizes = [1000, 10000]
    temps = np.linspace(0.0, 1.0, 5)
    power_law_flags = [False, True]

    for num_agents, temp, power in itertools.product(agent_sizes, temps, power_law_flags):

        config = SimConfig(
            num_agents=num_agents,
            mutation_temperature=temp,
            use_power_law_influence=power,
            seed=42
        )

        for name, fn in [("Discrete", discrete_fn), ("Density", density_fn)]:

            metrics, *_ = run_generator_with_metrics(fn, config)
            metrics["model"] = name
            metrics["num_agents"] = num_agents
            metrics["temperature"] = temp
            metrics["power_law"] = power

            results.append(metrics)

    return pd.DataFrame(results)


# ============================================================
# 4. DETERMINISM TEST (Case 7)
# ============================================================

def determinism_test(generator_fn, config, runs=3):

    outputs = []

    for _ in range(runs):
        df, exposures, personalities = generator_fn(config)
        outputs.append(torch.cat([exposures, personalities], dim=1))

    base = outputs[0]

    for i in range(1, runs):
        if not torch.equal(base, outputs[i]):
            return False

    return True


# ============================================================
# 5. SCALABILITY TEST (Case 6)
# ============================================================

def scalability_test(generator_fn):

    sizes = [1000, 5000, 10000]
    timings = []

    for n in sizes:
        config = SimConfig(num_agents=n, seed=42)
        start = time.time()
        generator_fn(config)
        elapsed = time.time() - start
        timings.append(elapsed)

    return list(zip(sizes, timings))


# ============================================================
# 6. VISUALIZATION (Cases 8 & 9)
# ============================================================

def compare_distributions(discrete_fn, density_fn):

    config = SimConfig(num_agents=10000, seed=42, use_power_law_influence=True)

    df_d, exp_d, _ = discrete_fn(config)
    df_g, exp_g, _ = density_fn(config)

    wealth_idx = DIMENSIONS.index("Wealth")

    wealth_d = exp_d[:, wealth_idx].numpy()
    wealth_g = exp_g[:, wealth_idx].numpy()

    plt.figure(figsize=(12,5))
    sns.histplot(wealth_d, label="Discrete", color="blue", stat="density", bins=50)
    sns.histplot(wealth_g, label="Density", color="red", stat="density", bins=50)
    plt.legend()
    plt.title("Wealth Distribution Comparison")
    plt.show()


# ============================================================
# 7. FINAL EVALUATION WRAPPER (Case 10)
# ============================================================

def full_evaluation(discrete_fn, density_fn):

    print("Running Parameter Grid Benchmark...")
    grid_results = recursive_grid_benchmark(discrete_fn, density_fn)

    print("Testing Determinism...")
    config = SimConfig(num_agents=5000, seed=123)
    det_discrete = determinism_test(discrete_fn, config)
    det_density = determinism_test(density_fn, config)

    print("Running Scalability Tests...")
    scale_discrete = scalability_test(discrete_fn)
    scale_density = scalability_test(density_fn)

    print("Generating Distribution Comparison Plot...")
    compare_distributions(discrete_fn, density_fn)

    return {
        "grid_results": grid_results,
        "determinism_discrete": det_discrete,
        "determinism_density": det_density,
        "scalability_discrete": scale_discrete,
        "scalability_density": scale_density,
    }

In [ ]:
results = full_evaluation(
    generate_society,
    generate_society_density_v2
)

results["grid_results"].head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set(style="whitegrid")


# ============================================================
# 1. AGGREGATE STATISTICS
# ============================================================

def summarize_results(results_df):

    summary = (
        results_df
        .groupby(["model", "num_agents", "temperature", "power_law"])
        .agg(["mean", "min", "max", "std"])
    )

    print("\n===== AGGREGATED METRICS =====\n")
    display(summary)

    return summary


# ============================================================
# 2. GINI INTERPRETATION
# ============================================================

def interpret_gini(gini_value):

    print("\n===== GINI INTERPRETATION =====\n")
    print(f"Gini Coefficient: {round(gini_value, 3)}")

    if gini_value < 0.3:
        print("→ Very low inequality (near egalitarian society)")
    elif gini_value < 0.5:
        print("→ Moderate inequality (typical developed economy range)")
    elif gini_value < 0.7:
        print("→ High inequality (strong concentration of wealth)")
    else:
        print("→ Extreme inequality (oligarchic concentration)")

    print("\nIn your simulation:")
    print("- Higher Gini = stronger wealth concentration.")
    print("- Power law influence will increase Gini.")
    print("- Density model may smooth class boundaries but not reduce structural inequality unless role weights shift.")


# ============================================================
# 3. BOX PLOTS (Outlier Detection)
# ============================================================

def boxplot_metrics(results_df):

    plt.figure(figsize=(14,6))
    sns.boxplot(data=results_df, x="model", y="gini_wealth")
    plt.title("Gini Distribution by Model")
    plt.show()

    plt.figure(figsize=(14,6))
    sns.boxplot(data=results_df, x="model", y="dna_drift")
    plt.title("DNA Drift Distribution by Model")
    plt.show()


# ============================================================
# 4. HISTOGRAMS
# ============================================================

def histogram_metrics(results_df):

    plt.figure(figsize=(12,5))
    sns.histplot(results_df["gini_wealth"], bins=20, kde=True)
    plt.title("Histogram of Gini Coefficients")
    plt.show()

    plt.figure(figsize=(12,5))
    sns.histplot(results_df["dna_drift"], bins=20, kde=True)
    plt.title("Histogram of DNA Drift")
    plt.show()


# ============================================================
# 5. LINE PLOTS (Temperature Effects)
# ============================================================

def temperature_effects(results_df):

    plt.figure(figsize=(12,6))
    sns.lineplot(
        data=results_df,
        x="temperature",
        y="dna_drift",
        hue="model",
        style="power_law",
        markers=True
    )
    plt.title("DNA Drift vs Temperature")
    plt.show()

    plt.figure(figsize=(12,6))
    sns.lineplot(
        data=results_df,
        x="temperature",
        y="gini_wealth",
        hue="model",
        style="power_law",
        markers=True
    )
    plt.title("Gini vs Temperature")
    plt.show()


# ============================================================
# 6. SCALABILITY CURVE
# ============================================================

def scalability_plot(scale_results, title):

    sizes = [x[0] for x in scale_results]
    times = [x[1] for x in scale_results]

    plt.figure(figsize=(8,5))
    plt.plot(sizes, times, marker="o")
    plt.title(title)
    plt.xlabel("Number of Agents")
    plt.ylabel("Execution Time (sec)")
    plt.show()


# ============================================================
# 7. CLUSTER VISUALIZATION (PCA Projection)
# ============================================================

def cluster_visualization(generator_fn):

    config = SimConfig(num_agents=5000, seed=42)
    df, exposures, personalities = generator_fn(config)

    traits = torch.cat([exposures, personalities], dim=1).numpy()

    scaled = StandardScaler().fit_transform(traits)
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(scaled)

    plt.figure(figsize=(8,6))
    sns.scatterplot(
        x=reduced[:,0],
        y=reduced[:,1],
        hue=df["Role"],
        palette="tab10",
        alpha=0.5,
        s=10
    )
    plt.title("PCA Projection of Social Landscape")
    plt.legend(bbox_to_anchor=(1.05,1))
    plt.show()


# ============================================================
# 8. MASTER ANALYSIS FUNCTION
# ============================================================

def advanced_analysis(results_dict):

    results_df = results_dict["grid_results"]

    summarize_results(results_df)

    # Interpret first Gini value
    interpret_gini(results_df["gini_wealth"].mean())

    boxplot_metrics(results_df)
    histogram_metrics(results_df)
    temperature_effects(results_df)

    print("\n===== SCALABILITY ANALYSIS =====\n")
    scalability_plot(
        results_dict["scalability_discrete"],
        "Discrete Model Scalability"
    )
    scalability_plot(
        results_dict["scalability_density"],
        "Density Model Scalability"
    )

    print("\n===== CLUSTER VISUALIZATION =====\n")
    cluster_visualization(generate_society)
    cluster_visualization(generate_society_density_v2)

    print("\n===== DETERMINISM =====")
    print("Discrete:", results_dict["determinism_discrete"])
    print("Density:", results_dict["determinism_density"])

In [ ]:
advanced_analysis(results)

density based models are more inclined

====New Methods====

In [ ]:
# ============================================================
# SYNTHETI-SOC FULL COLAB VERSION
# Complete generator + full SocietyEvolution logic
# No file I/O. Everything in-memory.
# ============================================================

import math
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import List, Dict

# ============================================================
# GLOBAL CONSTANTS
# ============================================================

DIMENSIONS: List[str] = [
    "Wealth","Physical_Safety","Stability","Reputation","Fairness",
    "In_Group","Innovation","Freedom","Sanctity","Care",
    "Short_Term","Long_Term",
]

REGIONS = ["US", "EU", "APAC", "Global_South"]
REGION_WEIGHTS = [0.15, 0.15, 0.35, 0.35]

REGION_ROLE_PROBS = {
    "US": [0.02,0.45,0.10,0.20,0.18,0.05],
    "EU": [0.01,0.48,0.08,0.25,0.15,0.03],
    "APAC": [0.005,0.60,0.05,0.10,0.20,0.045],
    "Global_South": [0.001,0.65,0.02,0.05,0.25,0.029],
}

ROLES = ["CEO","Worker","Investor","Retiree","Student","Politician"]

ROLE_STARTING_STATS = {
    "CEO": (50.0, 18.0),
    "Politician": (70.0, 25.0),
    "Investor": (25.0, 12.0),
    "Retiree": (10.0, 2.0),
    "Worker": (5.0, 1.0),
    "Student": (1.0, 0.6),
}

ARCHETYPES = {
    "CEO": {"exp": torch.tensor([0.8,0.1,0.3,0.9,0.1,0.4,0.9,0.8,0.1,0.1,0.7,0.9]),
            "big5": torch.tensor([0.8,0.9,0.8,0.3,0.3])},
    "Worker": {"exp": torch.tensor([0.4,0.8,0.9,0.3,0.7,0.5,0.2,0.4,0.5,0.6,0.9,0.2]),
               "big5": torch.tensor([0.4,0.6,0.5,0.6,0.7])},
    "Investor": {"exp": torch.tensor([0.9,0.1,0.2,0.6,0.2,0.3,0.8,0.8,0.0,0.1,0.8,0.6]),
                 "big5": torch.tensor([0.7,0.8,0.5,0.2,0.4])},
    "Retiree": {"exp": torch.tensor([0.3,0.9,1.0,0.2,0.5,0.8,-0.3,0.3,0.8,0.7,0.4,0.1]),
                "big5": torch.tensor([0.2,0.5,0.4,0.7,0.8])},
    "Student": {"exp": torch.tensor([0.1,0.3,0.2,0.5,0.8,0.3,0.9,0.7,0.1,0.8,0.6,0.8]),
                "big5": torch.tensor([0.9,0.3,0.7,0.6,0.5])},
    "Politician": {"exp": torch.tensor([0.6,0.5,0.8,1.0,0.4,0.9,0.3,0.6,0.6,0.5,1.0,0.2]),
                   "big5": torch.tensor([0.6,0.7,0.9,0.5,0.4])},
}

# ============================================================
# CONFIG
# ============================================================

@dataclass
class SimConfig:
    seed: int = 42
    num_agents: int = 10000
    mutation_temperature: float = 0.7

    enable_evolution: bool = True
    evolution_generations: int = 20

    inheritance_fraction: float = 0.7
    inheritance_noise_std: float = 0.05
    base_return_rate: float = 0.03
    influence_reinvestment_factor: float = 0.1
    reinvestment_noise_std: float = 0.02
    shock_frequency: float = 0.1
    shock_magnitude: float = 0.2
    mobility_rate: float = 0.05
    role_temperature: float = 0.5
    use_dynamic_roles: bool = True
    record_history: bool = True

# ============================================================
# MUTATION
# ============================================================

def apply_random_mutations(exposures, personalities, temperature, seed):
    if temperature <= 0:
        return exposures, personalities

    torch.manual_seed(seed + 999)
    n = exposures.shape[0]
    mask = torch.rand(n) < temperature
    idx = torch.where(mask)[0]

    if len(idx) == 0:
        return exposures, personalities

    num_changes = math.ceil(3 * temperature)

    for _ in range(num_changes):
        cols = torch.randint(0, exposures.shape[1], (len(idx),))
        exposures[idx, cols] = torch.rand(len(idx))*2 - 1

    for _ in range(num_changes):
        cols = torch.randint(0, personalities.shape[1], (len(idx),))
        personalities[idx, cols] = torch.rand(len(idx))

    return exposures, personalities

# ============================================================
# WEALTH ENGINE
# ============================================================

def generate_hybrid_wealth(role_wealth_bases, influence_scores, temperature, seed):
    np.random.seed(seed+123)
    n = len(role_wealth_bases)

    rank = np.argsort(np.argsort(influence_scores))
    pct = rank/(n-1)

    wealth = np.zeros(n)

    lower = pct < 0.60
    middle = (pct >= 0.60) & (pct < 0.95)
    upper = pct >= 0.95

    wealth[lower] = np.random.exponential(1+temperature, lower.sum())
    wealth[middle] = np.random.lognormal(0,0.5+temperature,middle.sum())
    alpha = max(1.1,1.5-temperature*0.3)
    wealth[upper] = (np.random.pareto(alpha,upper.sum())+1)*5

    wealth *= 1 + (pct**2)*(1+temperature)
    wealth *= role_wealth_bases
    return wealth

# ============================================================
# SOCIETY EVOLUTION (FULL FEATURED)
# ============================================================

class SocietyEvolution:

    def __init__(self, config, metadata, exposures, personalities):
        self.config = config
        self.metadata = metadata.copy()
        self.exposures = exposures.clone()
        self.personalities = personalities.clone()

        self.num_agents = len(self.metadata)
        self.wealth_idx = DIMENSIONS.index("Wealth")

        self.influence = torch.tensor(
            self.metadata["Influence"].values,
            dtype=torch.float32
        )

        self.history = {
            "wealth":[self.exposures[:,self.wealth_idx].clone()],
            "influence":[self.influence.clone()]
        }

    def apply_inheritance(self):
        inherit_frac = self.config.inheritance_fraction
        noise_std = self.config.inheritance_noise_std

        parent = self.exposures[:,self.wealth_idx]
        inherited = parent * inherit_frac
        noise = torch.randn(self.num_agents) * noise_std * parent.mean()

        self.exposures[:,self.wealth_idx] = torch.clamp(inherited+noise,min=0)

    def apply_reinvestment(self):
        base = self.config.base_return_rate
        factor = self.config.influence_reinvestment_factor
        noise_std = self.config.reinvestment_noise_std

        returns = base + factor*(self.influence/self.influence.mean())
        returns += torch.randn(self.num_agents)*noise_std
        returns = torch.clamp(returns,min=-0.2,max=0.5)

        self.exposures[:,self.wealth_idx] *= (1+returns)

    def apply_economic_shocks(self, gen):
        if np.random.rand() < self.config.shock_frequency:
            shock = 1 - self.config.shock_magnitude*np.random.uniform(0.5,1.0)
            print(f"Shock @ Gen {gen} → x{shock:.2f}")
            self.exposures[:,self.wealth_idx] *= shock

    def apply_mobility(self):
        n = int(self.num_agents*self.config.mobility_rate)
        idx = np.random.choice(self.num_agents,n,replace=False)
        shuffled = np.random.permutation(idx)

        new_inf = self.influence.clone()
        new_wealth = self.exposures[:,self.wealth_idx].clone()

        new_inf[idx] = self.influence[shuffled]
        new_wealth[idx] = self.exposures[shuffled,self.wealth_idx]

        self.influence = new_inf
        self.exposures[:,self.wealth_idx] = new_wealth

    def reassign_roles(self):
        wealth = self.exposures[:,self.wealth_idx]

        w_rank = torch.argsort(torch.argsort(wealth))
        w_norm = w_rank.float()/(self.num_agents-1)

        i_rank = torch.argsort(torch.argsort(self.influence))
        i_norm = i_rank.float()/(self.num_agents-1)

        power = 0.5*w_norm + 0.4*i_norm + 0.1*self.personalities.mean(dim=1)

        centers = torch.tensor([0.1,0.3,0.5,0.75,0.95])
        fitness = -((power.unsqueeze(1)-centers)**2)
        probs = torch.softmax(fitness/self.config.role_temperature,dim=1)

        elite_mask = w_norm < 0.95
        probs[elite_mask,4] = 0
        probs = probs/probs.sum(dim=1,keepdim=True)

        new_roles = torch.multinomial(probs,1).squeeze()
        self.metadata["Role"] = new_roles.numpy()

    def evolve(self):
        wealth_decay = 0.01          # 1% decay per generation
        influence_decay = 0.02       # 2% influence cooling
        redistribution_rate = 0.01   # 1% wealth redistribution
        for gen in range(1,self.config.evolution_generations+1):
            self.apply_inheritance()
            self.apply_reinvestment()
            self.apply_economic_shocks(gen)
            self.apply_mobility()
            self.apply_mobility()

            # ---------------------------
            # 5. STRUCTURAL DECAY (NEW)
            # ---------------------------

            # --- Wealth Decay ---
            self.exposures[:, self.wealth_idx] *= (1 - wealth_decay)

            # --- Influence Decay ---
            self.influence *= (1 - influence_decay)

            # --- Redistribution ---
            redistribution = (
                self.exposures[:, self.wealth_idx] * redistribution_rate
            )

            pool = redistribution.sum()

            self.exposures[:, self.wealth_idx] -= redistribution
            self.exposures[:, self.wealth_idx] += pool / self.num_agents

            if self.config.use_dynamic_roles:
                self.reassign_roles()

            self.exposures[:,self.wealth_idx] = torch.clamp(
                self.exposures[:,self.wealth_idx],0,1e6
            )

            if self.config.record_history:
                self.history["wealth"].append(
                    self.exposures[:,self.wealth_idx].clone()
                )
                self.history["influence"].append(
                    self.influence.clone()
                )

        self.metadata["Influence"] = self.influence.numpy()
        return self.metadata, self.exposures, self.personalities, self.history

# ============================================================
# GENERATOR
# ============================================================

def generate_society_structured(config: SimConfig):

    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    means = torch.stack([
        torch.cat([ARCHETYPES[r]["exp"],ARCHETYPES[r]["big5"]])
        for r in ROLES
    ])

    regions = np.random.choice(REGIONS,config.num_agents,p=REGION_WEIGHTS)
    traits = torch.zeros(config.num_agents,17)

    for region in REGIONS:
        mask = torch.from_numpy(regions==region)
        count = mask.sum().item()
        if count==0: continue

        role_weights = torch.tensor(REGION_ROLE_PROBS[region])
        comp = torch.multinomial(role_weights,count,replacement=True)

        std = 0.15+config.mutation_temperature*0.1
        traits[mask] = means[comp] + torch.randn(count,17)*std

    exposures = traits[:,:12]
    personalities = traits[:,12:]

    exposures, personalities = apply_random_mutations(
        exposures,personalities,
        config.mutation_temperature,
        config.seed
    )

    dist = torch.cdist(traits,means)
    closest = torch.argmin(dist,dim=1)
    roles = np.array([ROLES[i] for i in closest])

    wealth_base = np.array([ROLE_STARTING_STATS[r][0] for r in roles])
    influence_base = np.array([ROLE_STARTING_STATS[r][1] for r in roles])

    influence = influence_base*(1+np.random.lognormal(
        0,0.5+config.mutation_temperature,config.num_agents))

    wealth = generate_hybrid_wealth(
        wealth_base,influence,
        config.mutation_temperature,
        config.seed
    )

    exposures[:,0] = torch.tensor(wealth).float()

    metadata = pd.DataFrame({
        "Agent_ID":range(config.num_agents),
        "Role":roles,
        "Region":regions,
        "Influence":np.round(influence,3)
    })

    if config.enable_evolution:
        evolver = SocietyEvolution(config,metadata,exposures,personalities)
        return evolver.evolve()

    return metadata, exposures, personalities

In [ ]:
# ============================================================
# SYNTHETI-SOC BIAS-FREE VERSION
# Complete generator + full SocietyEvolution logic
# No structural priors
# ============================================================

import math
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import List

# ============================================================
# GLOBAL CONSTANTS
# ============================================================

DIMENSIONS: List[str] = [
    "Wealth","Physical_Safety","Stability","Reputation","Fairness",
    "In_Group","Innovation","Freedom","Sanctity","Care",
    "Short_Term","Long_Term",
]

ROLES = ["CEO","Worker","Investor","Retiree","Student","Politician"]

# ============================================================
# CONFIG (UNCHANGED)
# ============================================================

@dataclass
class SimConfig:
    seed: int = 42
    num_agents: int = 10000
    mutation_temperature: float = 0.7

    enable_evolution: bool = True
    evolution_generations: int = 20

    inheritance_fraction: float = 0.7
    inheritance_noise_std: float = 0.05
    base_return_rate: float = 0.03
    influence_reinvestment_factor: float = 0.1
    reinvestment_noise_std: float = 0.02
    shock_frequency: float = 0.1
    shock_magnitude: float = 0.2
    mobility_rate: float = 0.05
    role_temperature: float = 0.5
    use_dynamic_roles: bool = True
    record_history: bool = True

# ============================================================
# MUTATION (UNCHANGED)
# ============================================================

def apply_random_mutations(exposures, personalities, temperature, seed):
    if temperature <= 0:
        return exposures, personalities

    torch.manual_seed(seed + 999)
    n = exposures.shape[0]
    mask = torch.rand(n) < temperature
    idx = torch.where(mask)[0]

    if len(idx) == 0:
        return exposures, personalities

    num_changes = math.ceil(3 * temperature)

    for _ in range(num_changes):
        cols = torch.randint(0, exposures.shape[1], (len(idx),))
        exposures[idx, cols] = torch.rand(len(idx))*2 - 1

    for _ in range(num_changes):
        cols = torch.randint(0, personalities.shape[1], (len(idx),))
        personalities[idx, cols] = torch.rand(len(idx))

    return exposures, personalities

# ============================================================
# WEALTH ENGINE (UNCHANGED)
# ============================================================

def generate_hybrid_wealth(role_wealth_bases, influence_scores, temperature, seed):
    np.random.seed(seed+123)
    n = len(role_wealth_bases)

    rank = np.argsort(np.argsort(influence_scores))
    pct = rank/(n-1)

    wealth = np.zeros(n)

    lower = pct < 0.60
    middle = (pct >= 0.60) & (pct < 0.95)
    upper = pct >= 0.95

    wealth[lower] = np.random.exponential(1+temperature, lower.sum())
    wealth[middle] = np.random.lognormal(0,0.5+temperature,middle.sum())
    alpha = max(1.1,1.5-temperature*0.3)
    wealth[upper] = (np.random.pareto(alpha,upper.sum())+1)*5

    wealth *= 1 + (pct**2)*(1+temperature)
    wealth *= role_wealth_bases
    return wealth

# ============================================================
# SOCIETY EVOLUTION (IDENTICAL)
# ============================================================

class SocietyEvolution:

    def __init__(self, config, metadata, exposures, personalities):
        self.config = config
        self.metadata = metadata.copy()
        self.exposures = exposures.clone()
        self.personalities = personalities.clone()

        self.num_agents = len(self.metadata)
        self.wealth_idx = DIMENSIONS.index("Wealth")

        self.influence = torch.tensor(
            self.metadata["Influence"].values,
            dtype=torch.float32
        )

        self.history = {
            "wealth":[self.exposures[:,self.wealth_idx].clone()],
            "influence":[self.influence.clone()]
        }

    def apply_inheritance(self):
        parent = self.exposures[:,self.wealth_idx]
        inherited = parent * self.config.inheritance_fraction
        noise = torch.randn(self.num_agents) * \
                self.config.inheritance_noise_std * parent.mean()

        self.exposures[:,self.wealth_idx] = torch.clamp(inherited+noise,min=0)

    def apply_reinvestment(self):
        returns = (
            self.config.base_return_rate +
            self.config.influence_reinvestment_factor *
            (self.influence/self.influence.mean())
        )

        returns += torch.randn(self.num_agents)*self.config.reinvestment_noise_std
        returns = torch.clamp(returns,min=-0.2,max=0.5)

        self.exposures[:,self.wealth_idx] *= (1+returns)

    def apply_economic_shocks(self, gen):
        if np.random.rand() < self.config.shock_frequency:
            shock = 1 - self.config.shock_magnitude*np.random.uniform(0.5,1.0)
            print(f"Shock @ Gen {gen} → x{shock:.2f}")
            self.exposures[:,self.wealth_idx] *= shock

    def apply_mobility(self):
        n = int(self.num_agents*self.config.mobility_rate)
        idx = np.random.choice(self.num_agents,n,replace=False)
        shuffled = np.random.permutation(idx)

        new_inf = self.influence.clone()
        new_wealth = self.exposures[:,self.wealth_idx].clone()

        new_inf[idx] = self.influence[shuffled]
        new_wealth[idx] = self.exposures[shuffled,self.wealth_idx]

        self.influence = new_inf
        self.exposures[:,self.wealth_idx] = new_wealth

    def reassign_roles(self):
        wealth = self.exposures[:,self.wealth_idx]

        w_rank = torch.argsort(torch.argsort(wealth))
        w_norm = w_rank.float()/(self.num_agents-1)

        i_rank = torch.argsort(torch.argsort(self.influence))
        i_norm = i_rank.float()/(self.num_agents-1)

        power = 0.5*w_norm + 0.4*i_norm + 0.1*self.personalities.mean(dim=1)

        centers = torch.tensor([0.1,0.3,0.5,0.75,0.95])
        fitness = -((power.unsqueeze(1)-centers)**2)
        probs = torch.softmax(fitness/self.config.role_temperature,dim=1)

        elite_mask = w_norm < 0.95
        probs[elite_mask,4] = 0
        probs = probs/probs.sum(dim=1,keepdim=True)

        new_roles = torch.multinomial(probs,1).squeeze()
        self.metadata["Role"] = new_roles.numpy()

    def evolve(self):
        wealth_decay = 0.01          # 1% decay per generation
        influence_decay = 0.02       # 2% influence cooling
        redistribution_rate = 0.01   # 1% wealth redistribution
        for gen in range(1,self.config.evolution_generations+1):
            self.apply_inheritance()
            self.apply_reinvestment()
            self.apply_economic_shocks(gen)
            self.apply_mobility()
            self.apply_mobility()

            # ---------------------------
            # 5. STRUCTURAL DECAY (NEW)
            # ---------------------------

            # --- Wealth Decay ---
            self.exposures[:, self.wealth_idx] *= (1 - wealth_decay)

            # --- Influence Decay ---
            self.influence *= (1 - influence_decay)

            # --- Redistribution ---
            redistribution = (
                self.exposures[:, self.wealth_idx] * redistribution_rate
            )

            pool = redistribution.sum()

            self.exposures[:, self.wealth_idx] -= redistribution
            self.exposures[:, self.wealth_idx] += pool / self.num_agents

            if self.config.use_dynamic_roles:
                self.reassign_roles()

            self.exposures[:,self.wealth_idx] = torch.clamp(
                self.exposures[:,self.wealth_idx],0,1e6
            )

            if self.config.record_history:
                self.history["wealth"].append(
                    self.exposures[:,self.wealth_idx].clone()
                )
                self.history["influence"].append(
                    self.influence.clone()
                )

        self.metadata["Influence"] = self.influence.numpy()
        return self.metadata, self.exposures, self.personalities, self.history

# ============================================================
# GENERATOR (BIAS-FREE)
# ============================================================

def generate_society_bias_free(config: SimConfig):

    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    # ---- Continuous random trait field ----
    traits = torch.randn(config.num_agents, 17)

    exposures = traits[:,:12]
    personalities = torch.sigmoid(traits[:,12:])

    exposures, personalities = apply_random_mutations(
        exposures, personalities,
        config.mutation_temperature,
        config.seed
    )

    # ---- Influence (no role-based anchoring) ----
    influence = np.random.lognormal(
        mean=1.0,
        sigma=0.5+config.mutation_temperature,
        size=config.num_agents
    )

    # ---- Wealth (no role multipliers) ----
    wealth_base = np.ones(config.num_agents)

    wealth = generate_hybrid_wealth(
        wealth_base,
        influence,
        config.mutation_temperature,
        config.seed
    )

    exposures[:,0] = torch.tensor(wealth).float()

    metadata = pd.DataFrame({
        "Agent_ID":range(config.num_agents),
        "Role":np.zeros(config.num_agents),  # placeholder
        "Region":np.zeros(config.num_agents),  # no regions
        "Influence":np.round(influence,3)
    })

    if config.enable_evolution:
        evolver = SocietyEvolution(config,metadata,exposures,personalities)
        return evolver.evolve()

    return metadata, exposures, personalities

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import time
import itertools
import tracemalloc
import numpy as np
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import entropy


# ============================================================
# CORE METRICS
# ============================================================

def gini_coefficient(x: np.ndarray):
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n - 1) * x)) / (n * np.sum(x) + 1e-9)


def shannon_entropy_distribution(arr):
    values, counts = np.unique(arr, return_counts=True)
    probs = counts / counts.sum()
    return entropy(probs)


def compute_rank_correlation(x1, x2):
    r1 = np.argsort(np.argsort(x1))
    r2 = np.argsort(np.argsort(x2))
    return np.corrcoef(r1, r2)[0, 1]


def elite_turnover_rate(history_wealth, top_pct=0.05):
    turnovers = []
    n = len(history_wealth[0])
    top_k = int(n * top_pct)

    for t in range(1, len(history_wealth)):
        prev_top = np.argsort(history_wealth[t-1])[-top_k:]
        curr_top = np.argsort(history_wealth[t])[-top_k:]
        overlap = len(set(prev_top).intersection(set(curr_top)))
        turnovers.append(1 - overlap / top_k)

    return np.mean(turnovers)


def stability_index(series):
    if len(series) < 5:
        return np.var(series)
    return np.var(series[-5:])


def boundary_violation_check(exposures_tensor, wealth_idx):
    non_wealth_mask = torch.ones(exposures_tensor.shape[1], dtype=torch.bool)
    non_wealth_mask[wealth_idx] = False

    non_wealth = exposures_tensor[:, non_wealth_mask]
    wealth = exposures_tensor[:, wealth_idx]

    violations_non_wealth = torch.sum(
        (non_wealth < -1) | (non_wealth > 1)
    ).item()

    violations_wealth = torch.sum(
        (wealth < -100) | (wealth > 1e6)
    ).item()

    return violations_non_wealth, violations_wealth


# ============================================================
# SINGLE RUN WITH FULL EVOLUTION TRACKING
# ============================================================

def run_model_with_evolution(generator_fn, config):

    tracemalloc.start()
    start_time = time.time()

    df, exposures, personalities, history = generator_fn(config)

    execution_time = time.time() - start_time
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    wealth_idx = DIMENSIONS.index("Wealth")
    wealth = exposures[:, wealth_idx].numpy()

    wealth_traj = [w.numpy() for w in history["wealth"]]

    gini_traj = [gini_coefficient(w) for w in wealth_traj]
    final_gini = gini_traj[-1]

    final_entropy = shannon_entropy_distribution(df["Role"])
    turnover = elite_turnover_rate(wealth_traj)
    stability = stability_index(gini_traj)

    rank_corr = compute_rank_correlation(
        wealth_traj[0],
        wealth_traj[-1]
    )

    v_nonwealth, v_wealth = boundary_violation_check(
        exposures,
        wealth_idx
    )

    metrics = {
        "execution_time_sec": execution_time,
        "peak_memory_mb": peak / 1024**2,
        "final_gini": final_gini,
        "final_role_entropy": final_entropy,
        "elite_turnover": turnover,
        "rank_correlation_initial_final": rank_corr,
        "gini_stability_index": stability,
        "boundary_violations_nonwealth": v_nonwealth,
        "boundary_violations_wealth": v_wealth,
    }

    return metrics, gini_traj


# ============================================================
# FULL GRID EVOLUTION TEST
# ============================================================

def full_grid_evolution_test(structured_fn, bias_free_fn):

    results = []

    agent_sizes = [2000, 5000]
    generations = [10, 30]
    temps = [0.3, 0.7]
    mobility_rates = [0.02, 0.08]

    for n, gen, temp, mobility in itertools.product(
        agent_sizes,
        generations,
        temps,
        mobility_rates
    ):

        for model_name, fn in [
            ("Structured", structured_fn),
            ("BiasFree", bias_free_fn)
        ]:

            config = SimConfig(
                num_agents=n,
                mutation_temperature=temp,
                evolution_generations=gen,
                mobility_rate=mobility,
                enable_evolution=True,
                seed=42
            )

            metrics, _ = run_model_with_evolution(fn, config)

            metrics["model"] = model_name
            metrics["agents"] = n
            metrics["generations"] = gen
            metrics["temperature"] = temp
            metrics["mobility"] = mobility

            results.append(metrics)

    return pd.DataFrame(results)


# ============================================================
# DETERMINISM TEST
# ============================================================

def determinism_test(generator_fn, config, runs=3):

    outputs = []

    for _ in range(runs):
        df, exposures, personalities, _ = generator_fn(config)
        outputs.append(torch.cat([exposures, personalities], dim=1))

    base = outputs[0]

    for i in range(1, runs):
        if not torch.equal(base, outputs[i]):
            return False

    return True


# ============================================================
# SCALABILITY TEST
# ============================================================

def scalability_test(generator_fn):

    sizes = [1000, 5000, 10000]
    timings = []

    for n in sizes:
        config = SimConfig(num_agents=n, enable_evolution=True, seed=42)
        start = time.time()
        generator_fn(config)
        elapsed = time.time() - start
        timings.append((n, elapsed))

    return timings


# ============================================================
# EVOLUTION TRAJECTORY VISUALIZATION
# ============================================================

def compare_gini_trajectories(structured_fn, bias_free_fn):

    config = SimConfig(
        num_agents=5000,
        evolution_generations=30,
        enable_evolution=True,
        seed=42
    )

    _, gini_structured = run_model_with_evolution(structured_fn, config)
    _, gini_biasfree = run_model_with_evolution(bias_free_fn, config)

    plt.figure(figsize=(10,5))
    plt.plot(gini_structured, label="Structured")
    plt.plot(gini_biasfree, label="BiasFree")
    plt.xlabel("Generation")
    plt.ylabel("Gini Coefficient")
    plt.title("Gini Evolution Over Time")
    plt.legend()
    plt.show()


# ============================================================
# MASTER EVALUATION WRAPPER
# ============================================================

def full_evaluation(structured_fn, bias_free_fn):

    print("Running Full Grid Evolution Benchmark...")
    grid_results = full_grid_evolution_test(
        structured_fn,
        bias_free_fn
    )

    print("Testing Determinism...")
    config = SimConfig(num_agents=3000, enable_evolution=True, seed=123)
    det_structured = determinism_test(structured_fn, config)
    det_biasfree = determinism_test(bias_free_fn, config)

    print("Running Scalability Tests...")
    scale_structured = scalability_test(structured_fn)
    scale_biasfree = scalability_test(bias_free_fn)

    print("Plotting Gini Trajectories...")
    compare_gini_trajectories(structured_fn, bias_free_fn)

    return {
        "grid_results": grid_results,
        "determinism_structured": det_structured,
        "determinism_biasfree": det_biasfree,
        "scalability_structured": scale_structured,
        "scalability_biasfree": scale_biasfree,
    }

In [ ]:
results = full_evaluation(
    generate_society_structured,
    generate_society_bias_free
)

results["grid_results"].groupby("model").mean()

the previous method of probability is better for the case, despite assumptions. it is also faster, less resource intensive and the evolution model congerges to the same output with less variance